# MicroDuck RL — Colab A100 Trainer

이 저장소 자체가 공식 `microduck_rl` 소스 snapshot입니다.

- A100 GPU
- 4096 parallel environments
- 목표 6000 PPO iterations
- Google Drive checkpoint backup
- 세션 종료 후 자동 resume
- 최초 64 env / 5 iteration smoke test

먼저 **Runtime → Change runtime type → A100 GPU**를 선택하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
], check=False)

## 1. 설정

In [ ]:
TASK_ID = "Mjlab-Velocity-Flat-MicroDuck"
EXPERIMENT_NAME = "velocity"
NUM_ENVS = 4096
TARGET_ITERS = 6000
SYNC_INTERVAL = 60
DRIVE_ROOT = "/content/drive/MyDrive/microduck-training"
SMOKE_TEST = True

## 2. 저장소 clone + 설치

In [ ]:
import os, shutil, subprocess

REPO_DIR = "/content/microduck"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run([
        "git", "clone",
        "https://github.com/namyikim/microduck.git",
        REPO_DIR,
    ], check=True)

subprocess.run(
    ["bash", f"{REPO_DIR}/scripts/setup_colab.sh"],
    check=True,
)

## 3. 학습 시작 / 자동 resume

In [ ]:
import os, subprocess

env = os.environ.copy()
env.update({
    "TASK_ID": TASK_ID,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "NUM_ENVS": str(NUM_ENVS),
    "TARGET_ITERS": str(TARGET_ITERS),
    "SYNC_INTERVAL": str(SYNC_INTERVAL),
    "DRIVE_ROOT": DRIVE_ROOT,
    "SMOKE_TEST": "1" if SMOKE_TEST else "0",
    "WANDB_MODE": env.get("WANDB_MODE", "offline"),
    "REPO_DIR": REPO_DIR,
})

subprocess.run(
    ["bash", f"{REPO_DIR}/scripts/train_colab.sh"],
    env=env,
    check=True,
)

## 4. Drive checkpoint 확인

In [ ]:
from pathlib import Path
import re

root = Path(DRIVE_ROOT) / "logs" / "rsl_rl" / EXPERIMENT_NAME
items = []
if root.exists():
    for p in root.rglob("model_*.pt"):
        m = re.fullmatch(r"model_(\d+)\.pt", p.name)
        if m:
            items.append((int(m.group(1)), p))
items.sort(key=lambda x: x[0])

if not items:
    print("아직 checkpoint가 없습니다.")
else:
    for n, p in items[-10:]:
        print(f"{n + 1:6d} iterations (approx.)  {p}")
    print("Latest:", items[-1][1])